In [0]:
%run "/Workspace/Users/wade.amit8895@outlook.com/Azure-Databricks-Projects/customer_billing/notebooks/utility/common_functions"

In [0]:
dbutils.widgets.text("file_name", "")
dbutils.widgets.text("pipeline_name", "customer_billing")
dbutils.widgets.text("run_id", "")
dbutils.widgets.text("unzip_path", "")


file_name = dbutils.widgets.get("file_name")
pipeline_name = dbutils.widgets.get("pipeline_name")
run_id = dbutils.widgets.get("run_id")
unzip_path = dbutils.widgets.get("unzip_path")

run_start_time = datetime.now()
task_name = "stage_3_archive_zip"

if not run_id:
    raise ValueError("run_id parameter is required")

if not file_name:
    raise ValueError("file_name parameter is required")

In [0]:
def adls_path_exists(path):
    normalized_path = path.rstrip("/")
    parent_path = normalized_path.rsplit("/", 1)[0]
    try:
        files = dbutils.fs.ls(parent_path)
        return any(
            item.path.rstrip("/") == normalized_path
            for item in files
        )
    except Exception:
        return False

In [0]:
archive_folder = ("abfss://inbound@adlsstorageaccount5858.dfs.core.windows.net/" "ziparchive/" + datetime.now().strftime("%Y%m%d_%H%M%S") + "/")
source_zip_path = input_folder + file_name
archive_zip_path = archive_folder + os.path.basename(file_name)

#Log process started
process_log("Start", table_name="ZIP_ARCHIVAL", layer="bronze", error_message=None)

try:
    #Create dated archive folder
    dbutils.fs.mkdirs(archive_folder)

    #Archive ZIP
    if not adls_path_exists(source_zip_path):
        raise FileNotFoundError(f"Original ZIP not found: {source_zip_path}")

    if adls_path_exists(archive_zip_path):
        raise FileExistsError(f"Archive destination already exists: {archive_zip_path}")

    dbutils.fs.mv(source_zip_path, archive_zip_path)
    print(f"ZIP archived successfully: {archive_zip_path}")

    # Cleanup extracted files after successful ZIP archival
    if unzip_path and dbutils.fs.rm(unzip_path.rstrip("/"), True):
        print(f"Extracted folder deleted successfully: {unzip_path}")
    else:
        print(f"Extracted folder not found or could not be deleted: {unzip_path}")

    # Delete only the current ZIP's extracted folder
    if adls_path_exists(unzip_path):
        dbutils.fs.rm(unzip_path, True)
        print(f"Extracted folder deleted: {unzip_path}")
    else:
        print(f"Extracted folder not found: {unzip_path}")

    #Log success
    process_log("Success", table_name="", layer="bronze", error_message=None)

except Exception as e:
    #Log failure
    process_log("Failed", table_name="", layer="bronze", error_message=str(e))
    raise